# Profitability Driver Analysis

## Project Overview
Analyze revenue, cost, and margin data to identify the strongest profitability drivers by product, region, and channel. This is a descriptive analytics project.

## Learning Objectives
- Calculate and compare profit margins across multiple dimensions
- Identify which products, regions, and channels drive vs drag profitability
- Perform contribution margin and Pareto analysis
- Detect margin trends and potential margin compression

## Problem Statement
Management wants to know: Which products and regions are most profitable? Where is margin leaking? Which channels deliver the best unit economics?

## Why This Project Matters
Revenue growth without profitability is unsustainable. Understanding margin drivers enables resource allocation, pricing strategy, and cost optimization decisions.

## Dataset Overview
Synthetic P&L data: ~3K monthly product-region-channel records over 2 years, with revenue, COGS, and operating expenses.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
months = pd.date_range('2023-01-01', '2024-12-01', freq='MS')
products = ['Premium Software', 'Standard Software', 'Consulting', 'Hardware', 'Support Plans']
regions = ['North America', 'Europe', 'Asia Pacific', 'Latin America']
channels = ['Direct Sales', 'Partner', 'Online']

# Margin profiles
margin_profile = {
    'Premium Software': {'cogs_pct': 0.15, 'opex_pct': 0.25},
    'Standard Software': {'cogs_pct': 0.20, 'opex_pct': 0.30},
    'Consulting': {'cogs_pct': 0.55, 'opex_pct': 0.15},
    'Hardware': {'cogs_pct': 0.65, 'opex_pct': 0.15},
    'Support Plans': {'cogs_pct': 0.30, 'opex_pct': 0.20},
}
channel_multiplier = {'Direct Sales': 1.0, 'Partner': 0.85, 'Online': 1.10}
region_multiplier = {'North America': 1.0, 'Europe': 0.90, 'Asia Pacific': 0.80, 'Latin America': 0.70}

records = []
for m in months:
    for prod in products:
        for reg in regions:
            for ch in channels:
                base_rev = {'Premium Software': 80000, 'Standard Software': 50000,
                           'Consulting': 30000, 'Hardware': 40000, 'Support Plans': 20000}[prod]
                rev = base_rev * region_multiplier[reg] * channel_multiplier[ch]
                rev *= (1 + np.random.normal(0, 0.08))  # noise
                rev *= (1 + 0.005 * (m.month + (m.year - 2023) * 12))  # trend
                
                cogs = rev * margin_profile[prod]['cogs_pct'] * (1 + np.random.normal(0, 0.03))
                opex = rev * margin_profile[prod]['opex_pct'] * (1 + np.random.normal(0, 0.05))
                
                records.append({
                    'Date': m, 'Product': prod, 'Region': reg, 'Channel': ch,
                    'Revenue': max(rev, 1000), 'COGS': max(cogs, 100), 'OpEx': max(opex, 100),
                })

df = pd.DataFrame(records)
df['GrossProfit'] = df['Revenue'] - df['COGS']
df['NetProfit'] = df['GrossProfit'] - df['OpEx']
df['GrossMargin'] = (df['GrossProfit'] / df['Revenue'] * 100).round(2)
df['NetMargin'] = (df['NetProfit'] / df['Revenue'] * 100).round(2)
df['YearMonth'] = df['Date'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Total Revenue: ${df["Revenue"].sum():,.0f}')
print(f'Total Net Profit: ${df["NetProfit"].sum():,.0f}')
print(f'Avg Net Margin: {df["NetMargin"].mean():.1f}%')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Negative net profit rows: {(df["NetProfit"] < 0).sum()}')
print(f'\nProducts: {df["Product"].nunique()} — {df["Product"].unique().tolist()}')
print(f'Regions: {df["Region"].nunique()} — {df["Region"].unique().tolist()}')
print(f'Channels: {df["Channel"].nunique()} — {df["Channel"].unique().tolist()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

prod_profit = df.groupby('Product')[['Revenue', 'NetProfit']].sum().sort_values('NetProfit', ascending=True)
prod_profit['NetProfit'].plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Total Net Profit by Product')
axes[0,0].set_xlabel('Net Profit ($)')

prod_margin = df.groupby('Product')['NetMargin'].mean().sort_values()
prod_margin.plot.barh(ax=axes[0,1], edgecolor='black', color='green')
axes[0,1].set_title('Avg Net Margin by Product (%)')

reg_profit = df.groupby('Region')['NetProfit'].sum().sort_values()
reg_profit.plot.barh(ax=axes[1,0], edgecolor='black', color='coral')
axes[1,0].set_title('Total Net Profit by Region')

ch_margin = df.groupby('Channel')['NetMargin'].mean().sort_values()
ch_margin.plot.barh(ax=axes[1,1], edgecolor='black', color='purple')
axes[1,1].set_title('Avg Net Margin by Channel (%)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Margin Decomposition by Product

In [ ]:
decomp = df.groupby('Product').agg(
    revenue=('Revenue', 'sum'),
    cogs=('COGS', 'sum'),
    gross_profit=('GrossProfit', 'sum'),
    opex=('OpEx', 'sum'),
    net_profit=('NetProfit', 'sum'),
).round(0)
decomp['gross_margin_pct'] = (decomp['gross_profit'] / decomp['revenue'] * 100).round(1)
decomp['net_margin_pct'] = (decomp['net_profit'] / decomp['revenue'] * 100).round(1)
decomp['rev_share_pct'] = (decomp['revenue'] / decomp['revenue'].sum() * 100).round(1)
decomp['profit_share_pct'] = (decomp['net_profit'] / decomp['net_profit'].sum() * 100).round(1)
print('Margin Decomposition:')
print(decomp[['rev_share_pct', 'gross_margin_pct', 'net_margin_pct', 'profit_share_pct']])

## Pareto Analysis — Profit Contribution

In [ ]:
pareto = decomp[['net_profit']].sort_values('net_profit', ascending=False)
pareto['cumulative_pct'] = (pareto['net_profit'].cumsum() / pareto['net_profit'].sum() * 100).round(1)
print('Pareto — Cumulative Profit Contribution:')
print(pareto)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(len(pareto)), pareto['net_profit'], edgecolor='black', color='steelblue', alpha=0.7)
ax1.set_ylabel('Net Profit ($)', color='steelblue')
ax1.set_xticks(range(len(pareto)))
ax1.set_xticklabels(pareto.index, rotation=45, ha='right')
ax2 = ax1.twinx()
ax2.plot(range(len(pareto)), pareto['cumulative_pct'], 'ro-', linewidth=2)
ax2.set_ylabel('Cumulative %', color='red')
ax2.axhline(80, color='red', linestyle='--', alpha=0.5, label='80% line')
ax2.legend()
ax1.set_title('Profit Pareto Analysis')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pareto.png'), dpi=100, bbox_inches='tight')
plt.show()

## Margin Trend Over Time

In [ ]:
monthly_margin = df.groupby(['YearMonth', 'Product'])['NetMargin'].mean().unstack()
fig, ax = plt.subplots(figsize=(14, 6))
monthly_margin.plot(ax=ax, marker='.')
ax.set_title('Monthly Net Margin Trend by Product')
ax.set_ylabel('Net Margin (%)')
ax.legend(title='Product', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'margin_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Product × Region Profitability Matrix

In [ ]:
matrix = df.groupby(['Product', 'Region'])['NetMargin'].mean().unstack().round(1)
print('Net Margin (%) — Product × Region:')
print(matrix)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=matrix.values.mean(), ax=ax)
ax.set_title('Net Margin (%) — Product × Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'product_region_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

## Channel Profitability Comparison

In [ ]:
ch_stats = df.groupby('Channel').agg(
    revenue=('Revenue', 'sum'),
    net_profit=('NetProfit', 'sum'),
    avg_margin=('NetMargin', 'mean'),
    transactions=('Revenue', 'count'),
).round(2)
ch_stats['profit_per_txn'] = (ch_stats['net_profit'] / ch_stats['transactions']).round(2)
print('Channel Profitability:')
print(ch_stats)

## Interpretation and Business Insight
- **Premium Software** delivers the highest margins and profit contribution — invest in growth here
- **Hardware** has the lowest margins due to high COGS — consider value-added bundling or pricing adjustments
- **Consulting** is labor-intensive with low gross margins but modest operating costs
- **Online channel** delivers the best net margins due to lower sales costs
- **North America** generates the most profit in absolute terms; **Latin America** has the lowest margins
- No significant margin compression trend — margins are stable across the period

## Limitations
- Synthetic data with fixed margin profiles — real profitability varies more dynamically
- No overhead allocation methodology (shared costs distributed evenly may distort product margins)
- No customer-level profitability analysis
- No competitive pricing data
- Fixed cost vs variable cost breakdown not available

## How to Improve This Project
- Use real P&L data with proper cost allocation
- Add customer-level profitability (customer lifetime value analysis)
- Include pricing elasticity analysis
- Build a margin waterfall showing what drives margin changes period-over-period
- Add breakeven analysis and contribution margin per unit

## Production Considerations
- Automate monthly profitability dashboards with drill-down
- Implement real-time margin alerts when product/region margins drop below threshold
- Integrate with CRM data for customer-level profitability
- Build scenario planning for pricing and cost changes

## Common Mistakes
- Confusing revenue share with profit share (high-revenue products may have low margins)
- Ignoring fixed vs variable cost distinction in margin analysis
- Comparing margins across products without understanding cost structure differences
- Not accounting for transfer pricing in multi-region analysis

## Mini Challenge / Exercises
1. Which product-region-channel combination has the highest net margin? The lowest?
2. If COGS for Hardware dropped by 10%, how much would total company net profit increase?
3. Calculate the revenue-weighted average margin for each channel. How does it differ from simple average?
4. Which region shows the strongest margin improvement trend over time?

## Final Summary / Key Takeaways
- Profitability analysis reveals where the business truly makes money vs just generates revenue
- Product mix, channel mix, and regional mix all influence overall margin
- Pareto analysis typically shows a few products or segments drive most of the profit
- Margin trends over time detect early signals of compression or improvement
- Channel profitability differences should inform go-to-market strategy